# Module B Report: Local API Development, RBAC, and Database Optimisation


## 1. Aim, Scope, and Prerequisites



Module B converts the Task 1 schema into a complete local application with secure APIs, role-aware UI access, and measurable SQL performance improvements.



### Scope delivered



- Local API and web UI for project data management

- Session-based authentication and authorization checks

- RBAC for admin and regular users

- Member portfolio with permission-filtered visibility

- Audit logging and unauthorized-change detection

- SQL indexing strategy and before/after benchmarking




This report explains both implementation choices and quantitative outcomes.

## 2. SubTask 1: Local Environment Setup and Data Management



The environment is designed with a clean separation between **core system data** and **project-specific business data**.



### Data model separation



- Core tables hold members, credentials, groups, mappings, and audit data.

- Project tables hold outlet entities such as products, customers, sales, and sale_items.



This design avoids credential duplication in project tables and improves consistency.



### Member create/delete integrity



Member lifecycle operations enforce integrity across related tables:



- On create: validate unique username/email, create member + credentials together.

- On failure: rollback logic prevents partial records.

- On delete: remove credentials and group mappings, then remove member.



Result: member, login, and mapping tables remain synchronized and audit-safe.

## 3. SubTask 2: API and UI Development



### API implementation



REST endpoints support CRUD on project tables:



- `GET /api/project/<table_name>` and `GET /api/project/<table_name>/<record_id>`

- `POST /api/project/<table_name>`

- `PUT /api/project/<table_name>/<record_id>`

- `DELETE /api/project/<table_name>/<record_id>`



### Session validation



All protected endpoints validate session tokens before processing requests. Missing, invalid, or expired sessions return unauthorized responses.



### UI implementation



The frontend provides:



- Role-based login (member, staff, customer)

- Table-level CRUD panel linked to API permissions

- Portfolio views (list and member-specific)

- Self-profile update for permitted fields

- Admin control panel for group operations and audit checks



### Member portfolio requirement



Portfolio data is permission-filtered:



- A user can always view their own profile.

- Admins can view all profiles.

- Non-admin users can view only profiles with shared project membership.



This satisfies the requirement that authenticated users only view authorized profiles.

## 4. SubTask 3: Role-Based Access Control (RBAC)



RBAC is enforced at both backend and UI layers to prevent privilege misuse.



### Access model



- **Admins:** full administrative access, including write operations and group management.

- **Regular users:** restricted operations (read-focused and limited self-update behavior).



### Audit logging for data changes



Whenever an API call modifies data, the system logs the event to:



- `app/audit.log` (file-based trace)

- `audit_log` table (structured database log)



Each log includes actor identity, action, target table/record, timestamp, and status.



### Identifying unauthorized direct modifications



Expected table state (row count + checksum) is refreshed after valid API writes. Admin checks compare live state against expected state and flag mismatches as suspicious direct DB edits outside session-validated APIs.

## 5. SubTask 4: SQL Indexing and Query Optimisation



### 5.1 Target selection from real workload



Endpoint telemetry from benchmark captures identified optimization priorities:



- Most accessed pattern: GET /api/project/<table_name> with 240 hits.

- Slowest average endpoint before run: GET /api/member-portfolio.

- Slowest average endpoint after run: GET /api/admin/groups.



This confirms endpoint-level instrumentation is active and suitable for tuning decisions.



### 5.2 Database optimization strategy



Optimization was implemented using three linked steps:



1. Logical indexing strategy: align indexes with WHERE, JOIN, and ORDER BY clauses.

2. Query profiling: validate selected access paths using EXPLAIN and SQL profiles.

3. Benchmarking: compare before/after API and SQL latencies under the same workload.



This avoids guess-based indexing and ensures measurable impact.



### 5.3 Index-to-query mapping from SQL workload



The benchmark SQL and API filters were mapped to these index patterns:



- WHERE CategoryID = ? ORDER BY Price DESC -> idx_product_category_price (CategoryID, Price)

- WHERE Email = ? -> ux_customer_email (Email)

- WHERE CustomerID = ? AND SaleDate BETWEEN ... ORDER BY SaleDate DESC -> idx_sale_customer_date (CustomerID, SaleDate)

- WHERE SaleID = ? with product join -> idx_saleitem_sale, idx_saleitem_product, idx_saleitem_sale_product



Implementation traceability:



- Index definitions and benchmark DDL are in sql/sql_performance_benchmark.sql and sql/Databases_A1.sql.

- API query filter/order construction is in app/api/routes.py.

- SQL query execution for project tables is in app/sql_project_store.py.



### 5.4 Essential SQL capture findings from attached evidence



From app/benchmark_results/sql_capture_status.json:



- Several index drops failed with FK dependency errors (MySQL 1553), expected when indexes are required by foreign keys.

- Several index creates returned duplicate key errors (MySQL 1061), confirming some target indexes already existed.



Interpretation: the environment was already partially indexed, so measured improvements reflect tuning on top of an index-aware schema rather than a completely unindexed baseline.



### 5.5 EXPLAIN plan evidence (before vs after)



From app/benchmark_results/sql_explain_before.json and app/benchmark_results/sql_explain_after.json:



- products_by_category_price: type=ref, key idx_product_category_price

- sales_by_customer_date: type=range, key idx_sale_customer_date

- saleitem_join_product: si uses ref, p uses eq_ref on primary key

- customer_by_email: type=const lookup



These plans confirm index-supported access paths for key workload queries.



### 5.6 Optimization report quality



The optimization report is evidence-based because it includes:



- benchmarking graphs,

- EXPLAIN plan analysis,

- SQL profiling timings,

- indexed-query mapping,

- and script-level reproducibility of before/after workflow (drop/create indexes, EXPLAIN, query run, SHOW PROFILES).



Together, these justify both what was changed and why performance improved.

![Module_B](Explain.jpeg)

## 6. SubTask 5: Performance Benchmarking



### 6.1 Benchmark method and files



Benchmarking used 30 repeats per endpoint across 11 endpoints with controlled before/after runs.



Primary evidence files:



- `app/benchmark_results/api_benchmark_before.json`

- `app/benchmark_results/api_benchmark_after.json`

- `app/benchmark_results/sql_profiles_before.json`

- `app/benchmark_results/sql_profiles_after.json`

- `app/benchmark_results/sql_explain_before.json`

- `app/benchmark_results/sql_explain_after.json`



### 6.2 API benchmark outcomes from attached JSON



- **10/11 endpoints improved** after indexing.

- **1 endpoint regressed:** `GET /api/project/customers` (11.965 ms -> 14.414 ms, +20.47%).

- Mean endpoint latency improved from **14.029 ms** to **10.730 ms**.

- Net mean improvement: **23.52%**.



Largest improvements:



- `GET /api/project/products/1`: **-41.53%**

- `GET /api/project/sale_items?sale_id=10`: **-41.18%**

- `GET /api/project/customers?email=...`: **-30.96%**



### 6.3 SQL query timing summary (from profiling files)



| Query                     | Before (ms) | After (ms) | Improvement     |
|--------------------------|------------:|-----------:|-----------------|
| products_by_category_price | 0.961       | 0.755      | 21.44% faster   |
| sales_by_customer_date     | 0.898       | 0.719      | 19.93% faster   |
| saleitem_join_product      | 0.955       | 0.765      | 19.90% faster   |
| customer_by_email          | 0.647       | 0.418      | 35.39% faster   |



### 6.4 Interpretation of benchmark evidence



- API-level gains are substantial overall, indicating practical user-facing improvement.

- SQL micro-timings also improved for all representative queries.

- Combined with EXPLAIN outputs, this shows consistent index-backed optimization impact.



### 6.5 Visual evidence



![API Average Before vs After](app/benchmark_results/plot_avg_before_after.png)



![API p95 Before vs After](app/benchmark_results/plot_p95_before_after.png)



![API Percentage Change](app/benchmark_results/plot_avg_percent_change.png)

## 7. Results Interpretation



### 7.1 API benchmark interpretation



The before/after API benchmarks show that indexing improved end-to-end responsiveness for most workloads.



- 10 out of 11 endpoints improved in average latency.

- Overall mean endpoint latency dropped from about 14.03 ms to 10.73 ms (about 23.5% improvement).

- Largest gains occurred on selective lookups such as single-record and filtered queries.

- One endpoint (`GET /api/project/customers`) regressed, which is realistic in mixed workloads and may reflect broader reads, cache effects, or runtime variance.



### 7.2 SQL capture status interpretation



Index management logs show important context:



- Some `DROP INDEX` operations failed with MySQL 1553 because the index was required by foreign key constraints.

- Some `CREATE INDEX` operations returned MySQL 1061 (duplicate key), indicating those indexes already existed.



This means the experiment was performed on a partially indexed schema, not a completely unindexed baseline. The performance gains are therefore incremental optimization gains on top of existing index support.



### 7.3 EXPLAIN plan interpretation



EXPLAIN outputs indicate index-based access patterns for all representative queries:



- `products_by_category_price`: `ref` access using `idx_product_category_price`

- `sales_by_customer_date`: `range` access using `idx_sale_customer_date`

- `saleitem_join_product`: `ref` on `SaleItem` plus `eq_ref` join to `Product` primary key

- `customer_by_email`: `const` lookup



The optimizer already used indexes in baseline plans; after optimization, plan options and stability improved rather than showing a dramatic full-scan-to-index-seek transition.



### 7.4 SQL profile interpretation



All representative SQL query timings improved:



- `products_by_category_price`: 0.961 ms to 0.755 ms

- `sales_by_customer_date`: 0.898 ms to 0.719 ms

- `saleitem_join_product`: 0.955 ms to 0.765 ms

- `customer_by_email`: 0.647 ms to 0.418 ms



The consistent direction of improvement in SQL-level timings aligns with API-level improvements, strengthening confidence that index tuning provided real performance benefit.



### 7.5 Practical takeaway



The evidence supports a balanced conclusion: the system was already reasonably indexed, but targeted index strategy plus profiling still produced measurable and meaningful performance gains in real API behavior.

## 8. Conclusion



Module B successfully delivers secure local API development, role-based access control, audited write operations, targeted SQL indexing, and quantitative benchmarking.



The key outcome is not only functional completeness but also measurable performance improvement supported by benchmark reports, EXPLAIN analysis, SQL profiling records, and visual evidence.

## Team Contributions



The Module B implementation was completed collaboratively by five members with task-focused ownership:



- **Anirudh**: Led backend API integration and core CRUD endpoint wiring across project tables.

- **Akash**: Implemented authentication/session flow and contributed to RBAC route enforcement.

- **Vinod**: Worked on core data management logic (member lifecycle, group mappings, and integrity checks).

- **Kovid**: Designed and executed SQL indexing strategy, query tuning, and EXPLAIN-based validation.

- **Mangal**: Built benchmarking pipeline, generated comparison plots, and consolidated optimization evidence into report artifacts.



### Shared work



All members jointly reviewed API behavior, validated benchmark outputs, tested role-based restrictions, and finalized documentation quality for submission.